# 真实本地模型：CAA 激活转向（拟合与运行）

先用对比问答对拟合 CAA 转向向量（正负激活差的均值）并保存工件；再加载工件，在生成时把该向量加到指定层的输出上。

## 真实集成前提

安装：`python3 -m pip install -e ".[hf]"`。拟合需要 `prompt/positive/negative` 对比数据；本教程用仓库自带演示数据（仅教材示例规模，真实使用请准备领域数据）。

代码单元没有预执行。填写本地模型路径或 API 凭据后，从上到下运行；不要将 API key 保存到 notebook。缓解会改变模型输出，务必同时阅读基线与缓解后的回答。

## 参数说明

- `layer: int = 12`：注入转向向量的层。
- `multiplier: float = 1.0`：转向强度。
- `pooling: str = 'last'`：拟合时序列激活的池化方式（last/mean）。
- `chat=True`：数据按 chat 模板渲染。

## 步骤 1：配置真实模型与拟合数据

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

LOCAL_MODEL = "/absolute/path/to/your/causal-lm"   # e.g. a local Qwen2.5-1.5B-Instruct directory
DEVICE = "cpu"     # "cuda" when a GPU is available
DTYPE = "auto"
MAX_NEW_TOKENS = 64
if not Path(LOCAL_MODEL).is_dir():
    raise ValueError("Set LOCAL_MODEL to an existing local model directory (download it first).")

import json
DATA = repo_root / "examples" / "mitigation" / "data" / "hallucination_pairs_demo.jsonl"
ARTIFACT = repo_root / "runs" / "caa_demo.npz"
pairs = [json.loads(line) for line in DATA.read_text(encoding="utf-8").splitlines() if line.strip()]
print("pairs:", len(pairs), "| first keys:", sorted(pairs[0]))

## 步骤 2：拟合并保存工件

In [ ]:
from hallucination.mitigation.core.model_config import ModelConfig
from hallucination.mitigation.engine.hf import HFEngine
from hallucination.mitigation.methods.caa import CAAConfig, fit_caa
engine = HFEngine.from_pretrained(ModelConfig(model=LOCAL_MODEL, device=DEVICE, local_files_only=True))
artifact = fit_caa(engine, pairs, CAAConfig(layer=12, multiplier=1.0, chat=True, pooling="last"))
ARTIFACT.parent.mkdir(exist_ok=True)
artifact.save(str(ARTIFACT))
print("artifact:", ARTIFACT, "| layer:", artifact.layer)

## 步骤 3：加载工件交互运行

In [ ]:
from hallucination.mitigation import Mitigator
mitigator = Mitigator(
    method="caa",
    model=LOCAL_MODEL,
    device=DEVICE,
    dtype=DTYPE,
    local_files_only=True,
    steering=str(ARTIFACT),
    layer=12,
    multiplier=1.0,
    baseline=True,
)
prompt = input("Prompt：").strip()
report = mitigator.mitigate(prompt, max_new_tokens=MAX_NEW_TOKENS)
print("baseline :", report.baseline)
print("mitigated:", report.response)
print("warnings:", report.warnings)

## 结果解读与限制

CAA 可能过度修正（真实冒烟中把“小型甲壳类”改成“大型甲壳类”）；工件与模型结构绑定，跨模型复用工件会被拒绝；演示数据规模仅够跑通流程。